# Charger, adapter et configurer QLoRA pour le service client

Le fil rouge de la formation : le service client de Biclou & Cie, dont le modèle doit trier chaque message (catégorie, priorité) et y répondre selon la charte, en un objet JSON. Le chapitre suivant entraîne Qwen3-1.7B sur le jeu du TP de données. Ce TP prépare cet entraînement et en mesure le prix en mémoire, sur un modèle plus gros, Qwen3-4B-Instruct-2507 (4 milliards de paramètres) : ce que QLoRA fait tenir sur le GPU T4 de Colab.

1. Un modèle en 16, 8 puis 4 bits
2. LoRA léger ou ambitieux
3. Un entraînement QLoRA complet

Ce notebook demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il passe sur Qwen3-0.6B, chargé en float32 au lieu de 16 bits, avec un exemple par pas et un ou deux pas d'entraînement au lieu de dix ou vingt : le code tourne, mais ses mesures ne disent rien de celles d'un GPU.

## Données et modèle

`support-sft-train.jsonl` et `support-sft-validation.jsonl` : le jeu d'entraînement et le jeu de validation que produit le TP de données, au format conversationnel *prompt / completion* de `trl` (la consigne et le message du client dans `prompt`, le JSON attendu dans `completion`).

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/lab-results/support-sft-train.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/lab-results/support-sft-validation.jsonl
%pip install -q peft trl bitsandbytes
%pip uninstall -q -y torchao  # la version de Colab est trop ancienne pour peft

In [ ]:
import gc
import os
import time

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

GPU = torch.cuda.is_available()
# bfloat16 à partir des GPU Ampere (A100, L4, H100) ; le T4 calcule en float16.
BF16 = GPU and torch.cuda.get_device_capability()[0] >= 8
# Sur CPU, float32 : les demi-précisions y sont très lentes.
DTYPE = (torch.bfloat16 if BF16 else torch.float16) if GPU else torch.float32
DEVICE = "cuda" if GPU else "cpu"
MODELE = "Qwen/Qwen3-4B-Instruct-2507" if GPU else "Qwen/Qwen3-0.6B"
# Pas d'entraînement et exemples par pas : réduits sans GPU.
PAS = 10 if GPU else 1
LOT = 4 if GPU else 1
print(MODELE, "sur GPU" if GPU else "sans GPU")

tokenizer = AutoTokenizer.from_pretrained(MODELE)
entrainement = load_dataset("json", data_files="support-sft-train.jsonl", split="train")
validation = load_dataset(
  "json", data_files="support-sft-validation.jsonl", split="train"
)
if not GPU:
  entrainement = entrainement.select(range(8))
  validation = validation.select(range(4))
print(entrainement)


def memoire_go(modele) -> float:
  """La mémoire qu'occupe le modèle, en Go : celle du GPU, sinon celle de ses poids."""
  if GPU:
    return torch.cuda.memory_allocated() / 1e9
  return modele.get_memory_footprint() / 1e9


def pic_go() -> float:
  """Le pic de mémoire GPU depuis le dernier libere(), en Go (0 sans GPU)."""
  return torch.cuda.max_memory_allocated() / 1e9 if GPU else 0.0


def libere() -> None:
  """Rend au GPU la mémoire des objets supprimés par `del`, et remet le pic à zéro."""
  gc.collect()
  if GPU:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def repond(modele, conversation: list[dict]) -> str:
  """La réponse du modèle à une conversation, sans échantillonnage."""
  entree = tokenizer.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
    return_dict=True,
  ).to(DEVICE)
  with torch.no_grad():
    sortie = modele.generate(
      **entree,
      max_new_tokens=200,
      do_sample=False,
      temperature=None,
      top_p=None,
      top_k=None,
    )
  return tokenizer.decode(
    sortie[0, entree["input_ids"].shape[1] :], skip_special_tokens=True
  )

## 1. Un modèle en 16, 8 puis 4 bits

Écrivez `charge(quantification)`, qui charge `MODELE` sur `DEVICE` en `DTYPE`, quantifié selon `quantification` (une `BitsAndBytesConfig`, ou `None` pour le garder en 16 bits). Puis chargez-le quatre fois, en libérant la mémoire entre deux chargements : en 16 bits, en 8 bits, en NF4, en NF4 avec double quantification. Pour chacun, notez la mémoire (`memoire_go`) et la réponse au premier message de `validation` (`repond(modele, validation[0]["prompt"])`).

Comparez au calcul : 4 milliards de paramètres à 2, 1 et ½ octet. Pourquoi le modèle en 4 bits n'est-il pas 4 fois plus petit qu'en 16 bits ? La quantification a-t-elle abîmé la réponse ?

**Documentation utile**

- [`BitsAndBytesConfig`](https://huggingface.co/docs/transformers/main_classes/quantization#transformers.BitsAndBytesConfig)
- [Quantifier avec bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE, quantization_config=quantification)` : `quantization_config=None` charge le modèle tel quel
- 8 bits : `BitsAndBytesConfig(load_in_8bit=True)`
- NF4 : `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)`, et `bnb_4bit_use_double_quant=True` pour la double quantification
- Entre deux chargements : `del modele`, puis `libere()`, sinon les deux modèles s'additionnent sur le GPU
- Le premier chargement télécharge le modèle (8 Go) : quelques minutes ; les suivants le relisent du disque

```python
def charge(quantification: BitsAndBytesConfig | None):
  return AutoModelForCausalLM.from_pretrained(...)


def nf4(double: bool) -> BitsAndBytesConfig:
  return BitsAndBytesConfig(...)


chargements = {
  "16 bits": None,
  "8 bits": BitsAndBytesConfig(load_in_8bit=True),
  "NF4": nf4(double=False),
  "NF4, double quantification": nf4(double=True),
}
for nom, quantification in chargements.items():
  modele = charge(quantification)
  print(f"{nom} : {memoire_go(modele):.2f} Go")
  print(repond(modele, ...), "\n")
  del modele
  libere()
```

### Solution

In [ ]:
def charge(quantification: BitsAndBytesConfig | None):
  return AutoModelForCausalLM.from_pretrained(
    MODELE, dtype=DTYPE, device_map=DEVICE, quantization_config=quantification
  )


def nf4(double: bool) -> BitsAndBytesConfig:
  return BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=double,
    bnb_4bit_compute_dtype=DTYPE,
  )


chargements = {
  "16 bits": None,
  "8 bits": BitsAndBytesConfig(load_in_8bit=True),
  "NF4": nf4(double=False),
  "NF4, double quantification": nf4(double=True),
}
for nom, quantification in chargements.items():
  modele = charge(quantification)
  print(f"{nom} : {memoire_go(modele):.2f} Go")
  print(repond(modele, validation[0]["prompt"]), "\n")
  del modele
  libere()

# Les embeddings (151 936 tokens, 2 560 dimensions) restent en 16 bits : 0,8 Go
# dans les quatre cas, ce qui éloigne le modèle en 4 bits du quart du modèle en
# 16 bits. Sans GPU, get_memory_footprint() ne compte pas les constantes de NF4 :
# la double quantification n'y change rien.

## 2. LoRA léger ou ambitieux

Le modèle de base, chargé une fois en NF4 avec double quantification, reçoit tour à tour deux adaptateurs :

- léger : rang 8, sur les projections `q_proj` et `v_proj` de l'attention seulement (l'article de LoRA) ;
- ambitieux : rang 64, sur toutes les couches linéaires (`"all-linear"`).

Écrivez `essai(lora)`, qui entraîne l'adaptateur `lora` pendant `PAS` pas sur `entrainement`, avec `SFTTrainer`, puis renvoie le *trainer*. Pour chaque adaptateur, relevez le nombre de paramètres entraînés, le temps par pas, le pic de mémoire, la perte moyenne et la taille de l'adaptateur sauvegardé, puis rendez au modèle de base son état d'origine avant l'essai suivant. Lequel coûte le plus, et en quoi ? Que dit la perte après si peu de pas ?

**Documentation utile**

- [`LoraConfig`](https://huggingface.co/docs/peft/package_reference/lora#peft.LoraConfig)
- [`SFTTrainer` et `SFTConfig`](https://huggingface.co/docs/trl/sft_trainer)
- [`unload`](https://huggingface.co/docs/peft/package_reference/lora#peft.LoraModel.unload)

In [ ]:
base = charge(nf4(double=True))
libere()
print(f"Modèle de base : {memoire_go(base):.2f} Go")

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `SFTConfig(output_dir="essai", max_steps=PAS, per_device_train_batch_size=LOT, learning_rate=2e-4, gradient_checkpointing=True, bf16=BF16, fp16=GPU and not BF16, max_length=1024, logging_steps=1, save_strategy="no", report_to="none")` : `fp16` sur un T4, `bf16` sur un GPU plus récent, ni l'un ni l'autre sans GPU
- `SFTTrainer(model=base, args=config, train_dataset=entrainement, processing_class=tokenizer, peft_config=lora)` : le *trainer* ajoute l'adaptateur au modèle de base, et prépare le modèle en 4 bits à l'entraînement
- `trainer.model.get_nb_trainable_parameters()` renvoie (paramètres entraînés, paramètres en tout) ; `trainer.train().training_loss`, la perte moyenne
- `trainer.model.save_pretrained(dossier)`, puis la somme des `os.path.getsize` des fichiers de `os.listdir(dossier)`
- `trainer.model.unload()` retire l'adaptateur et rend le modèle de base, sans le modifier

```python
def essai(lora: LoraConfig) -> SFTTrainer:
  config = SFTConfig(...)
  return SFTTrainer(...)


def taille_mo(dossier: str) -> float:
  return (
    sum(os.path.getsize(os.path.join(dossier, f)) for f in os.listdir(dossier)) / 1e6
  )


adaptateurs = {
  "léger": LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"
  ),
  "ambitieux": LoraConfig(
    r=64, lora_alpha=128, target_modules="all-linear", task_type="CAUSAL_LM"
  ),
}
for nom, lora in adaptateurs.items():
  trainer = essai(lora)
  entraines, total = ...
  debut = time.time()
  perte = ...
  duree = (time.time() - debut) / PAS
  trainer.model.save_pretrained(nom)
  print(
    f"{nom} : {entraines / 1e6:.1f} M paramètres ({entraines / total:.2%}), "
    f"{duree:.1f} s par pas, pic {pic_go():.1f} Go, perte {perte:.3f}, "
    f"{taille_mo(nom):.0f} Mo"
  )
  base = ...
  del trainer
  libere()
```

### Solution

In [ ]:
def essai(lora: LoraConfig) -> SFTTrainer:
  config = SFTConfig(
    output_dir="essai",
    max_steps=PAS,
    per_device_train_batch_size=LOT,
    learning_rate=2e-4,
    gradient_checkpointing=True,
    bf16=BF16,
    fp16=GPU and not BF16,
    max_length=1024,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
  )
  return SFTTrainer(
    model=base,
    args=config,
    train_dataset=entrainement,
    processing_class=tokenizer,
    peft_config=lora,
  )


def taille_mo(dossier: str) -> float:
  return (
    sum(os.path.getsize(os.path.join(dossier, f)) for f in os.listdir(dossier)) / 1e6
  )


adaptateurs = {
  "léger": LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"
  ),
  "ambitieux": LoraConfig(
    r=64, lora_alpha=128, target_modules="all-linear", task_type="CAUSAL_LM"
  ),
}
for nom, lora in adaptateurs.items():
  trainer = essai(lora)
  entraines, total = trainer.model.get_nb_trainable_parameters()
  debut = time.time()
  perte = trainer.train().training_loss
  duree = (time.time() - debut) / PAS
  trainer.model.save_pretrained(nom)
  print(
    f"{nom} : {entraines / 1e6:.1f} M paramètres ({entraines / total:.2%}), "
    f"{duree:.1f} s par pas, pic {pic_go():.1f} Go, perte {perte:.3f}, "
    f"{taille_mo(nom):.0f} Mo"
  )
  base = trainer.model.unload()
  del trainer
  libere()

# Environ 40 fois plus de paramètres pour l'adaptateur ambitieux, mais un pic de
# mémoire à peine plus haut : les activations et le modèle de base dominent. Une
# perte sur dix pas ne départage pas deux configurations : l'évaluation du
# chapitre suivant le fera.

## 3. Un entraînement QLoRA complet

La configuration de l'entraînement du chapitre suivant, sur ce modèle de 4 milliards de paramètres : rang 16 sur toutes les couches linéaires, `lora_alpha` à 32, `lora_dropout` à 0,05 ; optimiseur AdamW 8 bits paginé, taux d'apprentissage 2e-4 avec une montée de 10 % des pas et une décroissance en cosinus ; *gradient checkpointing* ; *batch* de 4 exemples, accumulé sur 4 pas (16 exemples par mise à jour) ; évaluation sur `validation` à la fin. Écrivez-la, entraînez `2 * PAS` pas (sur GPU, un pas traite 16 exemples), et relevez le temps par pas, le pic de mémoire et la perte de validation.

Combien de temps durerait une époque complète sur le jeu d'entraînement de 1 190 exemples ? Que reste-t-il de libre sur les 15 Go du T4 ?

**Documentation utile**

- [`SFTConfig`](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig)
- [Optimiseurs paginés](https://huggingface.co/docs/bitsandbytes/optimizers#paged-optimizers)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")`
- Dans `SFTConfig` : `optim="paged_adamw_8bit"` (sur GPU seulement : `"adamw_torch"` sans GPU), `lr_scheduler_type="cosine"`, `warmup_steps=0.1` (sous 1 : une part des pas), `gradient_accumulation_steps=4`, `eval_strategy="steps"` et `eval_steps=2 * PAS` pour une évaluation au dernier pas
- `SFTTrainer(..., eval_dataset=validation, ...)` ; la perte de validation est dans `trainer.state.log_history`, une liste de dictionnaires : celui qui a une clé `"eval_loss"`
- Une époque : 1 190 exemples, à 16 par pas

```python
qlora = LoraConfig(...)
config = SFTConfig(
  output_dir="support-qlora",
  max_steps=2 * PAS,
  per_device_train_batch_size=LOT,
  gradient_accumulation_steps=...,
  optim=...,
  # les autres paramètres
  logging_steps=5,
  save_strategy="no",
  report_to="none",
)
trainer = SFTTrainer(...)
debut = time.time()
trainer.train()
duree = (time.time() - debut) / (2 * PAS)
perte = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h][-1]
pas_par_epoque = 1190 / 16
print(f"{duree:.1f} s par pas, pic {pic_go():.1f} Go, perte de validation {perte:.3f}")
print(
  f"Une époque : {pas_par_epoque:.0f} pas, {pas_par_epoque * duree / 60:.0f} minutes"
)
```

### Solution

In [ ]:
qlora = LoraConfig(
  r=16,
  lora_alpha=32,
  lora_dropout=0.05,
  target_modules="all-linear",
  task_type="CAUSAL_LM",
)
config = SFTConfig(
  output_dir="support-qlora",
  max_steps=2 * PAS,
  per_device_train_batch_size=LOT,
  gradient_accumulation_steps=4,
  learning_rate=2e-4,
  lr_scheduler_type="cosine",
  warmup_steps=0.1,
  optim="paged_adamw_8bit" if GPU else "adamw_torch",
  gradient_checkpointing=True,
  bf16=BF16,
  fp16=GPU and not BF16,
  max_length=1024,
  eval_strategy="steps",
  eval_steps=2 * PAS,
  per_device_eval_batch_size=LOT,
  logging_steps=5,
  save_strategy="no",
  report_to="none",
)
trainer = SFTTrainer(
  model=base,
  args=config,
  train_dataset=entrainement,
  eval_dataset=validation,
  processing_class=tokenizer,
  peft_config=qlora,
)
debut = time.time()
trainer.train()
duree = (time.time() - debut) / (2 * PAS)
perte = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h][-1]
pas_par_epoque = 1190 / 16
print(f"{duree:.1f} s par pas, pic {pic_go():.1f} Go, perte de validation {perte:.3f}")
print(
  f"Une époque : {pas_par_epoque:.0f} pas, {pas_par_epoque * duree / 60:.0f} minutes"
)